# Homework 3 — Colab Training

Train the **Classifier** and **Detector** on a Colab GPU (via the [Google Colab VS Code extension](https://marketplace.visualstudio.com/items?itemName=Google.colab) or colab.research.google.com).

## Quick start (VS Code + Colab extension)
1. Install the **Google Colab** extension.
2. Open this notebook and click **Select Kernel → Colab →** pick a **GPU** runtime (T4 is fine).
3. Run the cells top to bottom.

**Tips**
- Do **not** load datasets from Google Drive (very slow). Download into `/content` instead.
- Colab sessions wipe `/content` when they disconnect — re-download data and push/commit model weights (`*.th`) when done.
- After training, pull `homework/classifier.th` and `homework/detector.th` back to your local repo (git push from Colab, or download the files).

## 1. Check GPU

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU — enable a GPU runtime (Kernel → Colab → GPU).")

## 2. Get the homework code onto the Colab machine

The Colab kernel runs remotely, so your local files are **not** automatically available. Clone your GitHub repo (recommended), or upload `homework3/` manually.

For a **private** repo, set a classic PAT with `repo` scope:
```python
import os
os.environ["GITHUB_TOKEN"] = "ghp_..."
```

In [ ]:
import os
from pathlib import Path

repo_name = "online_deep_learning"
github_user = "henryingraham"  # change if needed
repo_path = Path(f"/content/{repo_name}")
hw_path = repo_path / "homework3"

# For a private repo, uncomment and paste a classic PAT with `repo` scope:
# os.environ["GITHUB_TOKEN"] = "ghp_..."

token = os.environ.get("GITHUB_TOKEN")
clone_url = (
    f"https://{token}@github.com/{github_user}/{repo_name}.git"
    if token
    else f"https://github.com/{github_user}/{repo_name}.git"
)

if not repo_path.exists():
    print(f"Cloning {clone_url}")
    !git clone {clone_url} {repo_path}
else:
    print("Repo already present — pulling latest")
    !git -C {repo_path} pull --ff-only

# Important: push your local homework3 code (models + train_*.py) to GitHub
# before cloning, otherwise Colab will train against stale/starter code.
assert (hw_path / "homework" / "models.py").exists(), f"Missing {hw_path}"
%ls {hw_path}

## 3. Install deps + enter `homework3/`

In [ ]:
%cd /content/online_deep_learning/homework3
!pip install -q -r requirements.txt

try:
    %load_ext autoreload
    %autoreload 2
except Exception as e:
    print("autoreload unavailable:", e)

%ls

## 4. Download datasets

Both classification and drive data are required.

In [ ]:
from pathlib import Path

if not Path("classification_data").exists():
    !curl -s -L https://www.cs.utexas.edu/~bzhou/dl_class/classification_data.zip -o classification_data.zip
    !unzip -qo classification_data.zip
    !rm classification_data.zip
else:
    print("classification_data already present")

if not Path("drive_data").exists():
    !curl -s -L https://www.cs.utexas.edu/~bzhou/dl_class/drive_data.zip -o drive_data.zip
    !unzip -qo drive_data.zip
    !rm drive_data.zip
else:
    print("drive_data already present")

!ls -la classification_data drive_data

## 5. Sanity-check models

In [ ]:
from homework.models import debug_model

debug_model(batch_size=2)

## 6. Train Classifier (Part 1)

Target: validation accuracy **> 0.80** (full credit). Extra credit toward **0.90**.

Best weights are saved to `homework/classifier.th`.

In [ ]:
from homework.train_classification import train as train_classifier

train_classifier(
    num_epoch=15,
    lr=1e-3,
    batch_size=128,
    seed=2024,
)

## 7. Train Detector (Part 2)

Targets:
- Segmentation IoU **> 0.75**
- Depth MAE **< 0.05**
- Lane-boundary depth MAE **< 0.05**

Best weights (by IoU) are saved to `homework/detector.th`.

In [ ]:
from homework.train_detection import train as train_detector

# Retrain after architecture/loss changes (old detector.th won't load).
# Depth is already easy; prioritize IoU via dice + stronger lane weights.
train_detector(
    num_epoch=30,
    lr=1e-3,
    batch_size=32,
    depth_weight=1.0,
    dice_weight=1.0,
    seed=2024,
)

## 8. Grade locally on Colab

In [ ]:
!python3 -m grader homework -v --disable_color

## 9. Bundle for Canvas (optional)

Replace `YOUR_UT_ID` with your UT EID, then download the zip from the Colab file browser.

In [ ]:
UT_ID = "YOUR_UT_ID"  # e.g. abc1234

!python3 bundle.py homework {UT_ID}
!python3 -m grader {UT_ID}.zip -v --disable_color
!ls -lh {UT_ID}.zip

## 10. Save weights back to GitHub (optional)

Push trained `.th` files so you can pull them locally.

In [ ]:
# Uncomment after setting GITHUB_TOKEN and verifying grader scores.
# !git -C /content/online_deep_learning config user.email "you@example.com"
# !git -C /content/online_deep_learning config user.name "Your Name"
# !git -C /content/online_deep_learning add homework3/homework/classifier.th homework3/homework/detector.th
# !git -C /content/online_deep_learning commit -m "Add homework3 trained weights"
# !git -C /content/online_deep_learning push

!ls -lh homework/*.th